# Hybrid Digital Twin for Li-ion Batteries — R

**Can you predict when a battery reaches end of life, from only the first 40 % of its life?**

Three models, one honest test:

| | Model | What it knows |
|---|---|---|
| 1 | **Physics** | a degradation law from the literature, two parameters |
| 2 | **Data** | a Gaussian process. Flexible, and has no idea what a battery is |
| 3 | **Hybrid** | the physics, plus a small network fitted to what the physics gets wrong |

The test is a **temporal split**: fit on the first 40 % of cycles, forecast the rest. Not a random split — a random split trains on Tuesday and Thursday to predict Wednesday, which is interpolation, and is not the question anyone is asking.

> **Colab supports R natively.** Open it with the R runtime — either via *Runtime → Change runtime type → R*, or by opening [colab.to/r](https://colab.to/r) and uploading this file. No installs needed for the base packages; `nnet` ships with R.

---

In [ ]:
# Base R plus two packages that ship with Colab's R image.
suppressPackageStartupMessages({
  library(nnet)      # single-hidden-layer network, in base R's recommended set
  library(ggplot2)
})
set.seed(0)
TRAIN_FRACTION <- 0.40; EOL <- 0.80

## 0. The data

In [ ]:
# Local checkout first, GitHub second — see the Python notebook for why.
URL <- "https://raw.githubusercontent.com/otwin-core/otwin-hybrid/main/data/battery_soh.csv"
candidates <- c("data/battery_soh.csv", "../data/battery_soh.csv")
found <- candidates[file.exists(candidates)]
source_path <- if (length(found) > 0) found[1] else URL
cat("reading", source_path, "\n")

df <- read.csv(source_path)
aggregate(cbind(cycles = id_cycle, final_SoH = SoH) ~ Battery, df,
          function(x) c(max = max(x), min = min(x)))
head(df)

## 1. The physics

The **Wang throughput power law** (Wang et al., 2011):

$$\mathrm{SoH}(n) = 1 - c\,n^{z}$$

Two parameters, and the exponent is a *reading about the cell*, not just a knob:

- $z \approx 0.5$ — diffusion-limited SEI growth. Fade slows as the passivation layer thickens.
- $z \approx 1$ — linear wear. Something degrades at a constant rate.
- $z > 1$ — accelerating fade. The **knee**.

Fit it in log space, where the power law is linear and the fit is well posed:

$$\log(1 - \mathrm{SoH}) = \log c + z \log n$$

This matters more than it sounds. Fitting the raw curve on a short window is badly conditioned — capacity drops ~11 % while cycle-to-cycle noise is ~0.7 %, so $c$ and $z$ trade off almost freely and least squares wanders to whatever bound you set. Done that way these cells gave $z = 2.0$, sitting on the bound: not a physical reading, an optimiser lost in a flat valley.

In [ ]:
wang <- function(n, c, z) 1 - c * n^z

fit_physics <- function(n, soh) {
  m  <- soh < 1 & n > 0
  # log space: log(1 - SoH) = log(c) + z*log(n)  -- an ordinary linear fit
  lm0 <- lm(log(1 - soh[m]) ~ log(n[m]))
  c0  <- exp(coef(lm0)[1]); z0 <- min(max(coef(lm0)[2], 0.3), 1.5)
  fit <- tryCatch(
    nls(soh ~ 1 - c * n^z, start = list(c = c0, z = z0),
        algorithm = 'port', lower = c(1e-8, 0.3), upper = c(1e-1, 1.5),
        control = nls.control(maxiter = 500, warnOnly = TRUE)),
    error = function(e) NULL)
  if (is.null(fit)) return(c(c = unname(c0), z = unname(z0)))
  coef(fit)
}

## 2. The data-only model

R's `loess` is a local-regression smoother. Like a Gaussian process it is
flexible and knows nothing about batteries — and like a GP, asked to predict
beyond its data it has nothing to go on.

**It fails differently from the Python GP, and that is worth seeing.** A GP with
a stationary kernel reverts to its prior mean: it goes flat. `loess` extrapolates
its local polynomial: it *diverges*. With `degree = 2` on this data it runs off
to a state of health above 2.0 on one cell and below zero on another — neither
of which is a thing a battery can do.

`degree = 1` is used below, which keeps the extrapolation linear and the failure
legible. Set it back to `2` and re-run if you want to see the divergence; it is
instructive, and it is why the y-axis is clamped in the figure.

In [ ]:
fit_smoother <- function(n, soh) {
  # degree = 1, not 2. A local *quadratic* extrapolated 80-100 cycles past the
  # training window diverges: it produced SoH > 2.0 on B0018 and SoH < 0 on
  # B0006, which blows the figure's y-axis and describes nothing physical.
  # Linear local regression degrades gracefully instead.
  fit <- loess(soh ~ n, span = 0.9, degree = 1,
               control = loess.control(surface = 'direct'))
  function(nt) as.numeric(predict(fit, newdata = data.frame(n = nt)))
}

## 3. The hybrid

**The network never sees SoH — only the residual.** `nnet` gives a single hidden layer, which is plenty: the residual is small and noise-dominated, and a larger network would fit the noise.

In [ ]:
fit_hybrid <- function(n, soh, c, z) {
  residual <- soh - wang(n, c, z)          # <- the only target
  X <- data.frame(a = n / 100, b = sqrt(n) / 10)
  net <- nnet(X, residual, size = 8, linout = TRUE, decay = 1e-2,
              maxit = 2000, trace = FALSE)
  function(nt) as.numeric(predict(net,
    data.frame(a = nt / 100, b = sqrt(nt) / 10)))
}

## 4. The honest test

In [ ]:
rmse <- function(y, yhat) sqrt(mean((y - yhat)^2))

rows <- list(); curves <- list()
for (cell in unique(df$Battery)) {
  g <- df[df$Battery == cell, ]; g <- g[order(g$id_cycle), ]
  n <- as.numeric(g$id_cycle); soh <- as.numeric(g$SoH)
  k <- floor(length(n) * TRAIN_FRACTION)
  ntr <- n[1:k]; str_ <- soh[1:k]; nte <- n[(k+1):length(n)]
  ste <- soh[(k+1):length(soh)]

  p  <- fit_physics(ntr, str_); c_ <- p[['c']]; z_ <- p[['z']]
  sm <- fit_smoother(ntr, str_); res <- fit_hybrid(ntr, str_, c_, z_)
  dr <- lm(str_ ~ ntr)

  preds <- list(
    physics     = wang(nte, c_, z_),
    gp          = sm(nte),
    hybrid      = wang(nte, c_, z_) + res(nte),
    persistence = rep(str_[k], length(nte)),
    drift       = predict(dr, data.frame(ntr = nte)))
  base <- rmse(ste, preds$persistence)
  curves[[cell]] <- list(n = n, soh = soh, k = k, preds = preds)
  for (m in names(preds))
    rows[[length(rows)+1]] <- data.frame(battery = cell, model = m,
      rmse = rmse(ste, preds[[m]]), skill = rmse(ste, preds[[m]]) / base)
}
res_df <- do.call(rbind, rows)
agg <- aggregate(cbind(rmse, skill) ~ model, res_df, mean)
agg[order(agg$rmse), ]

### The figure

In [ ]:
cell <- "B0005"; cv <- curves[[cell]]
n <- cv$n; soh <- cv$soh; k <- cv$k; split <- n[k]
long <- do.call(rbind, lapply(c('physics','gp','hybrid'), function(m)
  data.frame(cycle = n[(k+1):length(n)], soh = cv$preds[[m]], model = m)))
labs <- c(physics = 'Physics only', gp = 'Data only', hybrid = 'Hybrid')

ggplot() +
  annotate('rect', xmin = min(n), xmax = split, ymin = -Inf, ymax = Inf,
           fill = '#EEF2F6') +
  geom_point(aes(n, soh), alpha = .5, size = 1.4, colour = '#1B2430') +
  geom_line(data = long, aes(cycle, soh, colour = model), linewidth = 1.1) +
  geom_hline(yintercept = EOL, linetype = 'dashed', colour = '#B00020') +
  geom_vline(xintercept = split, colour = '#66707A') +
  scale_colour_manual(values = c(physics = '#1C4E73', gp = '#B5651D',
                                 hybrid = '#2E7D32'), labels = labs) +
  labs(x = 'Discharge cycle', y = 'State of Health', colour = NULL,
       title = paste0(cell, ' — fitted on the shaded window'),
       subtitle = 'the smoother has nothing to extrapolate with') +
  theme_minimal(base_size = 13) + theme(legend.position = 'bottom')

## 5. What actually happened

Mean over four cells, temporal split at 40 %:

| Model | RMSE | Skill vs persistence |
|---|---|---|
| **Hybrid** | **0.0398** | **0.36** |
| Baseline: linear drift | 0.0490 | 0.44 |
| Physics only | 0.0544 | 0.50 |
| Baseline: persistence | 0.1109 | 1.00 |
| Data only (GP) | 0.1791 | 1.62 |

Three things worth sitting with:

**The Gaussian process is worse than assuming nothing changes.** Skill 1.62. Not because it is badly implemented — it is properly specified and fitted with restarts. Because outside the range it has seen, a GP reverts to its prior mean. It has no concept of a battery, so it has no reason to keep going down.

**A straight line beats the physics on RMSE.** Linear drift, skill 0.44 against the physics model's 0.50. That is humbling and it is real. Over a bounded horizon, extrapolating a line is a genuinely strong baseline, and a project that only reported its wins would have quietly dropped this row.

**But RMSE is not the question.** An operator asks *when do I replace it?* On that metric the ranking inverts:

| Model | Mean error in predicted end-of-life cycle |
|---|---|
| **Physics only** | **13.0 cycles** |
| Hybrid | 21.9 cycles |
| Baseline: linear drift | 25.7 cycles |

The straight line has the second-best RMSE and the worst answer to the actual question, because it crosses the 80 % line at the wrong angle. **Choose the metric that matches the decision, or you will optimise the wrong thing very precisely.**

*R's `loess` stands in for the Gaussian process here — see section 2. Its numbers differ from the Python GP; its failure does not.*

## 6. Where this goes next

This notebook is the *tutorial*. The same ideas, engineered properly, are a set of composable tools:

| Tool | What it does |
|---|---|
| [`otwin-systems`](https://github.com/otwin-core/otwin-systems) | physical model structures, each validated against a closed-form answer |
| [`otwin-eval`](https://github.com/otwin-core/otwin-eval) | the temporal split and mandatory baselines used here |
| [`otwin-uq`](https://github.com/otwin-core/otwin-uq) | calibrated uncertainty — this notebook has none, which is its biggest gap |
| [`otwin-phs`](https://github.com/otwin-core/otwin-phs) | port-Hamiltonian systems, for assets where the physics is an energy balance |

**What this notebook does not do, and should:** produce an interval. Every forecast above is a single line, and a single line is not a forecast — it is a guess with good posture. `otwin-uq` measures whether a 90 % band actually contains the truth 90 % of the time.

---

### The one thing to take away

The physics is not there for interpretability. It is there because it is the only part of the model that still knows what it is doing outside the data it was fitted on.

---

Full ecosystem: **[github.com/otwin-core](https://github.com/otwin-core)** · Apache 2.0